# ***Recolección de datos: Reseñas y comentarios de usuarios de Nequi***

## ¿Qué haremos?
Construiremos la base de reseñas y comentarios que alimentará el proyecto de investigación. Descargaremos opiniones públicas de usuarios de **Nequi**, la billetera digital de Bancolombia, desde dos fuentes:

| Fuente | ¿Qué aporta? | Rol en el proyecto |
| --- | --- | --- |
| **Google Play** (Principal) | Texto, estrellas (1 - 5), "Me gusta", versión de la app y respuesta oficial de Nequi | Base del análisis de sentimiento y temas |

## **¿Por qué solo Google Play?**
Descartamos YouTube porque sus comentarios dependen del título y del enfoque de cada video. Eso introduce un sesgo de encuadre: el usuario reacciona al video, no a su experiencia con la app. Las reseñas de la tienda(En este caso: Google Play), en cambio, nacen de un uso real del producto.

## ¿Por qué construir una base propia?
Buscamos una base pública de comentarios sobre Nequi (En sitios como: Kaggle, repositorios académicos, GitHub) y *no encontramos ninguna*. Las bases disponibles tratan sobre otras aplicaciones o nichos de negocio. La mayoría están en inglés y/o son sintéticas.

Por eso decidimos **construir nuestra propia base** con **web scraping** sobre datos públicos. Esto supone tres ventajas:
1. **Datos reales y recientes** (Últimos 12 meses) de una marca Colombiana relevante.
2. **Control total del proceso:** Sabemos de dónde viene cada comentario y qué decisiones se toman al respecto.
3. **Una referencia para validar y comparar:** Las estrellas de Google Play permiten revisar posteriormente la clasificación automática de sentimiento con la dada por el usuario.

## Decisiones de recolección
- **País e idioma** Colombia (`co`) y español (`es`)
- **Ventana de tiempo:** Últimos 12 meses. Del más reciente al más antiguo
- **Tamaño máximo:** 6.000 reseñas de Google Play.
- **Privacidad:** No se guardan nombres ni fotos de usuarios; solo el texto y sus metadatos.
- **Archivo crudo congelado:** Se guarda una vez en `data/raw` y nunca se edita. La limpieza ocurre en el notebook `02_reseñas_usuarios_nequi_NLP-KMeans`

> Este notebook se ejecuta **UNA SOLA VEZ**. Si se vuelve a correr: Las reseñas cambiarán, esto porque Google Play siempre muestra primero las más recientes.

# Índice del notebook

| Índice | ¿Qué hace? |
| --- | --- |
| 1 | Instala las librerías y carga |
| 1.2 | Importaciones, rutas y parámetros |
| 1.3 | Ficha de la app en Google Play |
| 1.4 | Descarga de reseñas de Google Play |
| 1.5 | Exportación a `.xlsx`|

# 1 - Instalación y Drive

In [ ]:
# Instalar librerías de recolección (Versiones fijas para que el resultado sea reproducible)
! pip install -q google-play-scraper==1.2.7

# Montar Google Drive para guardar el dataset
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.2 - Importaciones, rutas y parámetros

In [ ]:
# Importaciones globales
import time
from datetime import datetime, timedelta
from itertools import islice
from pathlib import Path

import pandas as pd
from google_play_scraper import app, reviews, Sort

In [ ]:
# Rutas (El archvio crudo se escribe una vez y no se modifica)
RUTA_PROYECTO = Path("/content/drive/MyDrive/Colab Notebooks/No supervisado/Workshops/Final - Reseñas Usuarios Nequi")
DIR_CRUDOS = RUTA_PROYECTO / "data" / "raw"
DIR_CRUDOS.mkdir(parents = True, exist_ok = True)
ARCHIVO_CRUDO = DIR_CRUDOS / "nequi_reseñas_raw.xlsx"

# Parámetros de recolección
ID_APP = "com.nequi.MobileApp" # Nequi Colombia en Google Play
PAIS, IDIOMA = "co", "es" # Tienda Colombiana, reseñas en español
MAX_RESENAS = 35000   # ~85 reseñas/día × 365 días
FECHA_CORTE = datetime.now() - timedelta(days = 365) # Solo los últimos 12 meses
print("Fecha de corte:", FECHA_CORTE.date())

Fecha de corte: 2025-11-04


## 1.3 - Ficha de la app

## **Contexto de negocio**: Calificación promedio, volumen de reseñas e instalaciones

In [ ]:
##¿ Netadatos públicos de la app en la tienda colombiana
ficha = app(ID_APP, lang = IDIOMA, country = PAIS)
print(f"{ficha ['title']} | calificación: {ficha ['score']:.2f} | calificaciones: {ficha ['ratings']:,}"
      f"| reseñas: {ficha ['reviews']:,} | instalaciones: {ficha ['installs']}")

Nequi Colombia | calificación: 4.77 | calificaciones: 4,152,244| reseñas: 447,534 | instalaciones: 50.000.000+


## 1.4 - Reseñas de Google Play
Se descargan en páginas de a 200, de la más reciente a la más antigua, hasta llegar a la fecha corte o al tope

In [ ]:
# Paginar reseñas hasta pasar la fecha de corte o llegar al tope
filas, token = [], None
while len(filas) < MAX_RESENAS:
  lote, token = reviews(ID_APP, lang = IDIOMA, country = PAIS, sort = Sort.NEWEST,
                        count = 200, continuation_token = token)

  if not lote:
    break
  filas.extend(lote)
  print(f"Descargadas: {len(filas):,} | más antigua hasta ahora: {lote[-1]['at']:%Y-%m-%d}")
  if lote [-1]['at'] < FECHA_CORTE: # Ya saldríamos de la ventana de 12 meses
    break
  time.sleep(1) # Pausas para no saturar los servidores

gp = pd.DataFrame(filas)
gp = gp[gp['at'] >= FECHA_CORTE]

# Esquema común para todas las fuentes (sin nombres ni fotos de usuarios)
gp = pd.DataFrame({
    "fuente" : "Google Play",
    "id_comentario" : gp["reviewId"],
    "texto" : gp["content"],
    "calificacion": gp["score"], # #strellas 1-5: SOLO para validar el sentimiento
    "me_gusta": gp["thumbsUpCount"], # Votos de utilidad de otros usuarios
    "fecha": gp["at"],
    "version_app": gp["appVersion"],
    "respuesta_nequi" : gp["replyContent"],
    "fecha_respuesta" : gp["repliedAt"],
})
print(gp.shape, "|", gp["fecha"].min().date(), "->", gp["fecha"].max().date())

Descargadas: 200 | más antigua hasta ahora: 2026-11-01
Descargadas: 400 | más antigua hasta ahora: 2026-10-30
Descargadas: 600 | más antigua hasta ahora: 2026-10-27
Descargadas: 800 | más antigua hasta ahora: 2026-10-26
Descargadas: 1,000 | más antigua hasta ahora: 2026-10-23
Descargadas: 1,200 | más antigua hasta ahora: 2026-10-21
Descargadas: 1,400 | más antigua hasta ahora: 2026-10-19
Descargadas: 1,600 | más antigua hasta ahora: 2026-10-17
Descargadas: 1,800 | más antigua hasta ahora: 2026-10-15
Descargadas: 2,000 | más antigua hasta ahora: 2026-10-12
Descargadas: 2,200 | más antigua hasta ahora: 2026-10-09
Descargadas: 2,400 | más antigua hasta ahora: 2026-10-07
Descargadas: 2,600 | más antigua hasta ahora: 2026-10-04
Descargadas: 2,800 | más antigua hasta ahora: 2026-10-02
Descargadas: 3,000 | más antigua hasta ahora: 2026-09-29
Descargadas: 3,200 | más antigua hasta ahora: 2026-09-26
Descargadas: 3,400 | más antigua hasta ahora: 2026-09-24
Descargadas: 3,600 | más antigua hasta 

## 1.5 - Exportar reseñas a `.xlsx`

In [ ]:
# Quitar reseñas repetidas y congelar el archivo crudo:
crudo = gp.drop_duplicates("id_comentario")
crudo.to_excel(ARCHIVO_CRUDO, index = False)

print("Reseñas guardadas:", len(crudo))
print("\nDistribución de estrellas:")
print(crudo["calificacion"].value_counts().sort_index())
print("\nGuardado en:", ARCHIVO_CRUDO)

Reseñas guardadas: 34328

Distribución de estrellas:
calificacion
1     6375
2     1324
3     1591
4     2854
5    22184
Name: count, dtype: int64

Guardado en: /content/drive/MyDrive/Colab Notebooks/No supervisado/Workshops/Final - Reseñas Usuarios Nequi/data/raw/nequi_reseñas_raw.xlsx


## 1.6 - **Hallazgos de los datos recolectados**

- **34.328 reseñas** de Google Play Colombia, del 04/11/2025 al 03/11/2026 (12 Meses completos, apróximadamente 94 reseñas recibidas por día)
- **Distribución en forma de J:** 64,6% con calificación de *5 estrellas* y 18,6% con calificación de *1 estrella*. Las calificaciones intermedias son escasas
- **49% de las reseñas tiene 3 palabras o menos**. Coinciden con caídas reportadas por la prensa (06/12/2025: 767, 81% negativas).
- **Limitaciones:** Solo usuarios de Andrior OS; las reseñas son autoseleccionadas (escribe más quien está muy satisfecho o muy molesto); `fecha` corresponde a la última edición de la reseña (No a la original). 10% de las reseñas sin versión de la app.